# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [2]:
# Install uv
#!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
#!$HOME/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
#!.venv/bin/python -m pip install sympy numpy transformers tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

#!.venv/bin/python -m pip install accelerate ipykernel jupyter

# Install Jupyter Kernel
#!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

#!.venv/bin/python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

Done. Restart the kernel before proceeding.
Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.


### Run the cell below every time to activate the installed environment. 

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [1]:
import sys, torch
print("Python path:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())

Python path: /home/eholguin/CSE151B/.venv/bin/python
Torch version: 2.5.1+cu121
CUDA: 12.1
GPU: True


## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
# from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [33]:
# SYSTEM_PROMPT_MATH = (
#     "You are an expert mathematician. Solve the problem step-by-step. "
#     "Put your final answer inside \\boxed{}. "
#     "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
#     "e.g. \\boxed{3, 7}."
# )

# SYSTEM_PROMPT_MCQ = (
#     "You are an expert mathematician. "
#     "Read the problem and the answer choices below, then select the single best answer. "
#     "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
# )


# def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
#     """Return (system_prompt, user_prompt) for a question."""
#     if options:
#         labels    = [chr(65 + i) for i in range(len(options))]
#         opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#         return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    #return SYSTEM_PROMPT_MATH, question


# # Verify with samples
# for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
#     sys_p, usr_p = build_prompt(item["question"], item.get("options"))
#     print(f"── {label} user prompt (first 200 chars) ──")
#     print(usr_p[:200], "...\n")


SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem carefully step by step. "
    "Put the final answer inside \\boxed{}. "
    "Do not write anything after the boxed answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Solve the multiple-choice problem carefully. "
    "Compare each answer choice. "
    "At the end, output ONLY the correct letter inside \\boxed{}, "
    "for example \\boxed{A}. "
    "Do not output any other final answer."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(
            f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
        )

        user_prompt = f"""{question}
            Options:
                {opts_text}

                Solve the problem carefully. The correct answer is one of the options above.
                Final answer must be exactly one boxed letter, like \\boxed{{F}}."""

        return SYSTEM_PROMPT_MCQ, user_prompt

    user_prompt = (
        f"Question:\n{question.strip()}\n\n"
        "Answer only with one boxed expression."
    )

    return SYSTEM_PROMPT_MATH, user_prompt



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
import torch
# import sys
# print(sys.executable)
# print(sys.version)
# print(torch.__version__)
# print(torch.__file__)
# import os
# print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
# print("is_available:", torch.cuda.is_available())
# print("device_count:", torch.cuda.device_count())


In [6]:

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cuda"   # ⭐ loads directly to GPU safely
)

model.eval()
print(next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

cuda:0


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [ ]:
# #}Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # # Generate
# #print(f"Generating responses for {len(prompts)} questions...")
# #outputs = llm.generate(prompts, sampling_params=sampling_params)

# #responses = [out.outputs[0].text.strip() for out in outputs]

# # # Preview first 3
# #for i in range(min(3, len(responses))):
#     #print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     #print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

# responses = []

# print(f"Generating responses for {len(prompts)} questions...")
# count = 1
# for prompt in prompts:
#     print(count)
    
#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=100,
#         do_sample=True,
#         temperature=0.7,
#     )
    
#     text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
#     # Optional: remove the prompt part
#     response = text[len(prompt):].strip()
#     responses.append(response)
#     count += 1

# ===== BUILD PROMPTS =====

eval_data = data[:5]   # change to data for full run

prompts = []

for item in eval_data:
    system, user = build_prompt(item["question"], item.get("options"))

    try:
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
            #enable_thinking=False,
        )
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )

    prompts.append(prompt_text)

# ===== EXTRACTION =====

def extract_boxed(text):
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", str(text), re.IGNORECASE)
    return boxed[-1].strip() if boxed else None


def extract_answer(text, is_mcq=False):
    text = str(text).strip()

    boxed = extract_boxed(text)
    if boxed is not None:
        if is_mcq:
            m = re.search(r"[A-F]", boxed.upper())
            return m.group(0) if m else boxed.upper()
        return boxed

    if is_mcq:
        letters = re.findall(r"\b([A-F])\b", text.upper())
        if letters:
            return letters[-1]

    frac = re.findall(r"-?\d+\s*/\s*-?\d+", text)
    if frac:
        return frac[-1].replace(" ", "")

    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    if nums:
        return nums[-1]

    return text[:50]

# ===== GENERATION =====

# raw_responses = []
# responses = []

# print(f"Generating responses for {len(prompts)} questions...")

# for i, prompt in enumerate(prompts):
#     print(f"\nQuestion {i+1}")

#     inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#     input_len = inputs["input_ids"].shape[1]

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=1024,
#         do_sample=False,   # IMPORTANT
#         pad_token_id=tokenizer.eos_token_id,
#     )

#     # decode ONLY generated tokens
#     new_tokens = outputs[0][input_len:]
#     raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

#     is_mcq = bool(eval_data[i].get("options"))
#     final_answer = extract_answer(raw_response, is_mcq)

#     raw_responses.append(raw_response)
#     responses.append(final_answer)

#     print("Raw:", raw_response[:300])
#     print("Extracted:", final_answer)
#     print("Gold:", eval_data[i]["answer"])
#     print("------")

# ===== MAJORITY VOTING GENERATION =====

from collections import Counter

NUM_VOTES = 3   # try 3 first; use 5 if you have time/GPU memory

raw_responses = []
responses = []

print(f"Generating responses for {len(prompts)} questions with {NUM_VOTES} votes each...")

# for i, prompt in enumerate(prompts):
#     print(f"\nQuestion {i+1}")

#     is_mcq = bool(eval_data[i].get("options"))

#     vote_answers = []
#     vote_raws = []

#     for vote in range(NUM_VOTES):
#         inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
#         input_len = inputs["input_ids"].shape[1]

#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=2048,
#             do_sample=True,
#             temperature=0.4,
#             top_p=0.9,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#         new_tokens = outputs[0][input_len:]
#         raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

#         final_answer = extract_answer(raw_response, is_mcq)

#         vote_raws.append(raw_response)
#         vote_answers.append(final_answer)

#         print(f"Vote {vote+1}: {final_answer}")

#     # Pick most common extracted answer
#     final_answer = Counter(vote_answers).most_common(1)[0][0]

#     # Save the raw response that produced the winning answer
#     winning_index = vote_answers.index(final_answer)
#     winning_raw = vote_raws[winning_index]

#     raw_responses.append(winning_raw)
#     responses.append(final_answer)

#     print("Votes:", vote_answers)
#     print("Chosen:", final_answer)
#     print("Gold:", eval_data[i]["answer"])
#     print("------")
def choose_mcq_by_logprob(question, options):
    labels = [chr(65 + i) for i in range(len(options))]

    opts_text = "\n".join(
        f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options)
    )

    # 🔥 FIX: force model to predict the answer token directly
    mcq_prompt = (
        f"Question:\n{question.strip()}\n\n"
        f"Options:\n{opts_text}\n\n"
        "The correct answer is \\boxed{"
    )

    inputs = tokenizer(mcq_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :]
        logprobs = torch.log_softmax(logits, dim=-1)

    scores = {}

    for L in labels:
        candidates = [L, " " + L]

        best_score = float("-inf")

        for cand in candidates:
            token_ids = tokenizer.encode(cand, add_special_tokens=False)

            if len(token_ids) == 1:
                score = logprobs[0, token_ids[0]].item()
                best_score = max(best_score, score)

        scores[L] = best_score

    print("MCQ scores:", scores)  # debug

    return max(scores, key=scores.get)
    
for i, prompt in enumerate(prompts):
    print(f"\nQuestion {i+1}")

    is_mcq = bool(eval_data[i].get("options"))

    # =========================
    # MCQ → use logprob scoring
    # =========================
    if is_mcq:
        final_answer = choose_mcq_by_logprob(
            eval_data[i]["question"],
            eval_data[i]["options"]
        )

        raw_response = f"\\boxed{{{final_answer}}}"

        raw_responses.append(raw_response)
        responses.append(final_answer)

        print("Chosen (logprob):", final_answer)
        print("Gold:", eval_data[i]["answer"])
        print("------")

    # =========================
    # FREE-FORM → majority voting
    # =========================
    else:
        vote_answers = []
        vote_raws = []

        for vote in range(NUM_VOTES):
            inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
            input_len = inputs["input_ids"].shape[1]

            outputs = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=True,
                temperature=0.5,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )

            new_tokens = outputs[0][input_len:]
            raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

            final_answer = extract_answer(raw_response, False)

            vote_raws.append(raw_response)
            vote_answers.append(final_answer)

            print(f"Vote {vote+1}: {final_answer}")

        # majority vote
        final_answer = Counter(vote_answers).most_common(1)[0][0]

        winning_index = vote_answers.index(final_answer)
        winning_raw = vote_raws[winning_index]

        raw_responses.append(winning_raw)
        responses.append(final_answer)

        print("Votes:", vote_answers)
        print("Chosen:", final_answer)
        print("Gold:", eval_data[i]["answer"])
        print("------")

Generating responses for 5 questions with 3 votes each...

Question 1
Vote 1: 325/2
Vote 2: 105950
Vote 3: 105950
Votes: ['325/2', '105950', '105950']
Chosen: 105950
Gold: ['325*(1+325)']
------

Question 2
MCQ scores: {'A': -3.927734375, 'B': -2.357421875, 'C': -3.849609375, 'D': -5.65625, 'E': -1.3427734375, 'F': -2.935546875, 'G': -5.5703125, 'H': -0.88916015625, 'I': -5.0, 'J': -3.060546875}
Chosen (logprob): H
Gold: F
------

Question 3
Vote 1: 8/11
Vote 2: 8/11
Vote 3: 8284/3
Votes: ['8/11', '8/11', '8284/3']
Chosen: 8/11
Gold: ['143.224229233795', '2.32624773420025']
------

Question 4


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [31]:
# # def extract_letter(text: str) -> str:
# #     m = re.search(r"\\boxed\{([A-Za-z])\}", text)
# #     if m:
# #         return m.group(1).upper()
# #     matches = re.findall(r"\b([A-Z])\b", text.upper())
# #     return matches[-1] if matches else ""

# def extract_letter(text: str) -> str:
#     text = str(text).strip().upper()

#     # If response is already just A/B/C/D/E/F
#     if re.fullmatch(r"[A-F]", text):
#         return text

#     # If response is boxed, like \boxed{C}
#     m = re.search(r"\\BOXED\{([A-F])\}", text)
#     if m:
#         return m.group(1)



# def score_mcq(response: str, gold_letter: str) -> bool:
#     return extract_letter(response) == gold_letter.strip().upper()


# # Load Judger for free-form scoring
# sys.path.insert(0, ".")
# from judger import Judger
# judger = Judger(strict_extract=False)

# results = []
# for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
#     is_mcq = bool(item.get("options"))
#     gold   = item["answer"]

#     if is_mcq:
#         correct = score_mcq(response, str(gold))
#     else:
#         gold_list = gold if isinstance(gold, list) else [gold]
#         try:
#             correct = judger.auto_judge(
#                 pred=response,
#                 gold=gold_list,
#                 options=[[]] * len(gold_list),
#             )
#         except Exception:
#             correct = False

#     results.append({
#         "id":       item.get("id"),
#         "is_mcq":   is_mcq,
#         "gold":     gold,
#         "response": response,
#         "correct":  correct,
#     })

# print(f"Scoring complete. {len(results)} results.")
# ===== SCORING =====

def score_mcq(response: str, gold_letter: str) -> bool:
    return str(response).strip().upper() == str(gold_letter).strip().upper()


sys.path.insert(0, ".")
from judger import Judger

judger = Judger(strict_extract=False)

results = []

for item, raw_response, response in tqdm(
    zip(eval_data, raw_responses, responses),
    total=len(eval_data),
    desc="Scoring"
):
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(response, gold)

    else:
        gold_list = gold if isinstance(gold, list) else [gold]

        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except:
            correct = False

        # fallback to raw output if needed
        if not correct:
            try:
                correct = judger.auto_judge(
                    pred=raw_response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except:
                correct = False

    results.append({
        "id": item.get("id"),
        "is_mcq": is_mcq,
        "gold": gold,
        "response": response,
        "correct": correct,
    })

print(f"Scoring complete. {len(results)} results.")


Scoring: 100%|██████████| 5/5 [00:00<00:00, 35.00it/s]

Scoring complete. 5 results.


## 8. Summary

Print accuracy broken down by question type.

In [32]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [15]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!